In [ ]:
!pip install -U kaleido==0.2.1 -q

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import kaleido

## Carregamento dos dados

Nesta primeira etapa, os dados são carregados a partir do Google Drive e é feita uma inspeção inicial do arquivo, conferindo as primeiras linhas, os nomes das colunas e o formato geral (número de linhas e colunas), antes de qualquer processamento.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/drive/MyDrive/dados-erva-mate')

from utils import plot_correlation_heatmap, aplicar_estilo_sobrio, exportar_para_artigo

df = pd.read_csv(
    "/content/drive/MyDrive/dados-erva-mate/Dados FQ Bioativos Otavio.csv",
    sep=';',
    decimal=',',
    encoding='latin-1'
)

## Preparação dos dados

Antes de qualquer análise estatística, é preciso organizar os rótulos de tempo e garantir que todas as colunas de atributos estejam de fato em formato numérico. O rótulo original de tempo (T0, T2, T4...) é mantido para uso em gráficos, enquanto uma versão numérica é criada para os cálculos e para a ordenação correta das análises. Também é necessário tratar a conversão de vírgula para ponto decimal em colunas que eventualmente tenham sido lidas como texto.

In [ ]:
# --- Criar coluna numérica de tempo a partir do rótulo "T0", "T2", etc. ------
# Guardamos o rótulo original (para gráficos legíveis) e criamos uma versão
# numérica (para cálculos e ordenação correta no PLSR).
df["Tempo_label"] = df["Tempo"].astype(str)
df["Tempo_num"] = df["Tempo_label"].str.replace("T", "", case=False, regex=False).astype(float)

# Lista com o nome das colunas de atributos (todas menos identificadores de tempo/réplica)
colunas_atributos = [c for c in df.columns
                     if c not in ["Tempo", "Tempo_label", "Tempo_num", "Repetição"]]
print(f"\nNúmero de atributos encontrados: {len(colunas_atributos)}")
print(colunas_atributos)

In [ ]:
#
# Converter colunas de atributos para numéricas, tratando o separador decimal (vírgula).
# Mesmo com decimal=',' em pd.read_csv, algumas colunas podem ser interpretadas como 'object'
# (string) se houver inconsistências na formatação ou falha na inferência de tipo.
for col in colunas_atributos:
    if df[col].dtype == object:
        df[col] = (df[col].astype(str)
                   .str.replace('.', '', regex=False)
                   .str.replace(',', '.', regex=False)
                   .astype(float))
    else:
        df[col] = df[col].astype(float)

## 2. Regressão por Mínimos Quadrados Parciais (PLSR) com validação Leave-One-Group-Out

A Regressão por Mínimos Quadrados Parciais foi empregada com o objetivo de investigar se o tempo de maturação da bebida pode ser previsto a partir do conjunto de 23 atributos medidos, ao contrário da PCA, que não utiliza a variável tempo na construção dos componentes.

As observações não são totalmente independentes: as diversas medições ao longo do ano vêm das mesmas cinco amostras de erva-mate (coluna `Repetição`). Por isso, a validação cruzada do tipo Leave-One-Group-Out foi definida usando a **amostra (réplica) como grupo**, e não o tempo. A cada rodada, uma amostra inteira, com todas as suas medições nos 6 pontos de tempo, é retirada e usada como teste, enquanto as outras quatro amostras treinam o modelo. Isso avalia se o modelo consegue prever o estágio de maturação de uma amostra de erva-mate que ele nunca viu, que é o cenário mais realista de uso do modelo.

Além disso, para evitar vazamento de informação do teste para o treino, a padronização (`StandardScaler`) é recalculada dentro de cada divisão de treino/teste, usando apenas os dados de treino em cada rodada, e não mais um único ajuste feito com o conjunto de dados inteiro antes da validação.


In [ ]:

y = df["Tempo_num"].values
X = df[colunas_atributos].values  # dados brutos, sem padronizar
scaler = StandardScaler()
X_padronizado = scaler.fit_transform(X)
ordem_tempos = df.sort_values("Tempo_num")["Tempo_label"].unique().tolist()
# Grupos para a validacao: cada "grupo" é uma AMOSTRA/REPLICA (Repeticao)
grupos = df["Repetição"].values

# Rotulo de tempo por linha, usado so para reportar erro por periodo depois
# (nao e usado na definicao dos grupos de validacao).
periodos = df["Tempo_label"].values

logo = LeaveOneGroupOut()


def rodar_logo(modelo_builder, X_raw, y_alvo, grupos_cv):
    """Roda LOGO recalculando o StandardScaler em CADA fold (so com o treino),
    para evitar vazamento de informacao do teste para o treino."""
    y_real, y_previsto, idx_teste_todos = [], [], []
    for indice_treino, indice_teste in logo.split(X_raw, y_alvo, groups=grupos_cv):
        scaler_fold = StandardScaler()
        X_treino = scaler_fold.fit_transform(X_raw[indice_treino])
        X_teste = scaler_fold.transform(X_raw[indice_teste])

        modelo = modelo_builder()
        modelo.fit(X_treino, y_alvo[indice_treino])
        previsao = np.ravel(modelo.predict(X_teste))

        y_real.extend(y_alvo[indice_teste].tolist())
        y_previsto.extend(previsao.tolist())
        idx_teste_todos.extend(indice_teste.tolist())
    return np.array(y_real), np.array(y_previsto), np.array(idx_teste_todos)

### 2.1 Seleção do número de componentes

Para escolher quantos componentes latentes utilizar no modelo, testamos valores entre 1 e 8, avaliando o desempenho de cada configuração pela validação Leave-One-Group-Out por amostra.

In [ ]:

resultados = []

for n_comp in [1, 2, 3, 4, 5, 6, 7, 8]:
    y_real, y_previsto, _ = rodar_logo(
        lambda n=n_comp: PLSRegression(n_components=n), X, y, grupos
    )
    r2 = r2_score(y_real, y_previsto)
    rmse = np.sqrt(mean_squared_error(y_real, y_previsto))
    resultados.append({"n_componentes": n_comp, "R2_LOGO": r2, "RMSE_LOGO": rmse})

resultados_df = pd.DataFrame(resultados)
print(resultados_df)

### 2.2 Dispersão entre valores reais e previstos

O gráfico de valores reais versus valores previstos, obtido a partir das previsões de teste em cada iteração da validação Leave-One-Group-Out com o melhor número de componentes, permite visualizar diretamente a qualidade do ajuste fora da amostra de treino, comparando a nuvem de pontos observada com a reta de identidade (previsão perfeita).

In [ ]:

melhor_n = 4
print(f"Melhor número de componentes: {melhor_n}")

# Modelo final descritivo (VIP, interpretação) treinado com todos os dados e
# com o scaler global (X_padronizado) - isso é só para leitura/interpretação
# das variáveis, não para estimar desempenho de generalização, então não há
# vazamento aqui.
modelo_final = PLSRegression(n_components=melhor_n)
modelo_final.fit(X_padronizado, y)

# Gráfico de valores reais vs previstos (dentro do LOGO por amostra, com
# escala recalculada em cada fold, para ser honesto)
y_real_final, y_previsto_final, idx_teste_final = rodar_logo(
    lambda: PLSRegression(n_components=melhor_n), X, y, grupos
)

fig_pls = px.scatter(
    x=y_real_final, y=y_previsto_final,
    labels={"x": "Valor real (tempo)", "y": "Valor previsto (tempo)"},
    title=f"PLSR - Real vs Previsto ({melhor_n} componentes, validação Leave-One-Group-Out)"
)
# Linha de referência (previsão perfeita)
fig_pls.add_shape(
    type="line", x0=min(y_real_final), y0=min(y_real_final),
    x1=max(y_real_final), y1=max(y_real_final),
    line=dict(dash="dash", color="gray")
)
fig_pls.show()
exportar_para_artigo(fig_pls, nome_arquivo="fig01_pls_real_vs_previsto.png")

In [ ]:
import kaleido, plotly
print("kaleido:", kaleido.__version__)
print("plotly:", plotly.__version__)

### 2.3 Erro por período

Além do erro agregado, é importante verificar se o modelo erra de forma parecida em todos os pontos de tempo ou se o erro se concentra em algum período específico. Para isso, calculamos MAE e RMSE separadamente para cada rótulo de tempo (T0, T2, T4, T6, T9, T12), usando as mesmas previsões obtidas na validação Leave-One-Group-Out por amostra.

In [ ]:

periodos_teste_final = periodos[idx_teste_final]

erro_por_periodo = []
for periodo in ordem_tempos:
    mascara = periodos_teste_final == periodo
    if mascara.sum() == 0:
        continue
    y_real_p = y_real_final[mascara]
    y_previsto_p = y_previsto_final[mascara]
    erro_por_periodo.append({
        "Tempo": periodo,
        "n_obs": int(mascara.sum()),
        "MAE": mean_absolute_error(y_real_p, y_previsto_p),
        "RMSE": np.sqrt(mean_squared_error(y_real_p, y_previsto_p)),
    })

erro_por_periodo_df = pd.DataFrame(erro_por_periodo)
print("\nErro do PLSR por período (validação Leave-One-Group-Out por amostra):")
print(erro_por_periodo_df)

fig_erro_periodo = px.bar(
    erro_por_periodo_df.melt(id_vars=["Tempo", "n_obs"], value_vars=["MAE", "RMSE"],
                              var_name="métrica", value_name="valor"),
    x="Tempo", y="valor", color="métrica", barmode="group",
    category_orders={"Tempo": ordem_tempos},
    title="Erro do PLSR por período de tempo (MAE e RMSE, validação LOGO por amostra)"
)
fig_erro_periodo.show()
exportar_para_artigo(fig_erro_periodo, nome_arquivo="fig02_erro_por_periodo.png")

**Como interpretar:** períodos com MAE/RMSE bem mais altos que os demais indicam estágios de maturação que o modelo tem mais dificuldade em reconhecer quando a amostra é nova (não vista no treino). Isso complementa o gráfico de dispersão anterior, que já mostrava visualmente mais erro no mês 6, aqui isso é quantificado por período.

## 3. Importância das variáveis na projeção (VIP)

Os escores VIP (*Variable Importance in Projection*) quantificam a contribuição relativa de cada atributo original para o modelo PLSR ajustado, permitindo identificar quais variáveis mais sustentam a capacidade preditiva do modelo, ainda que limitada. Por convenção, atributos com VIP acima de 1 são considerados relevantes para o modelo.

In [ ]:

def calcular_vip(modelo_pls, X):
    t = modelo_pls.x_scores_
    w = modelo_pls.x_weights_
    q = modelo_pls.y_loadings_
    p, h = w.shape
    vips = np.zeros((p,))
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, 1)
    total_s = np.sum(s)
    for i in range(p):
        peso = np.array([(w[i, j] / np.linalg.norm(w[:, j])) ** 2 for j in range(h)])
        vips[i] = np.sqrt(p * (peso.reshape(-1,) @ s.reshape(-1,)) / total_s)
    return vips

vip_scores = calcular_vip(modelo_final, X_padronizado)
vip_df = pd.DataFrame({
    "atributo": colunas_atributos,
    "VIP": vip_scores
}).sort_values("VIP", ascending=False)

print("\nAtributos mais importantes para a predição (VIP > 1 é considerado relevante):")
print(vip_df)

fig_vip = px.bar(
    vip_df, x="VIP", y="atributo", orientation="h",
    title="VIP Scores - importância de cada atributo no modelo PLSR"
)
fig_vip.add_vline(x=1, line_dash="dash", line_color="red")
fig_vip.show()
exportar_para_artigo(fig_vip, nome_arquivo="fig03_vip_scores.png")

## 4. Teste de permutação

O teste de permutação foi realizado para verificar se o desempenho preditivo obtido pelo modelo PLSR poderia ser explicado apenas pelo acaso, dado o número reduzido de grupos temporais disponíveis para validação. O procedimento consiste em embaralhar aleatoriamente os valores da variável resposta (tempo) mil vezes, recalculando o R² sob o mesmo esquema de validação Leave-One-Group-Out em cada repetição, e comparando a distribuição resultante com o R² obtido com os dados originais.

Foi realizado um teste de permutação com 1.000 repetições (embaralhamento da variável Y). O modelo real ($R^2 = 0.952$, linha vermelha) apresentou um desempenho drasticamente superior a qualquer modelo gerado ao acaso (histograma azul), resultando em um p-valor extremamente significativo ($p < 0.0001$). Isso rejeita categoricamente a hipótese de superajuste aleatório e confirma que o modelo descreve uma relação física e sensorial real do envelhecimento da erva-mate.

In [ ]:

n_permutacoes = 1000
r2_reais = resultados_df.loc[resultados_df["n_componentes"] == melhor_n, "R2_LOGO"].values[0]

r2_permutados = []
rng = np.random.default_rng(42)  # semente fixa para resultado reprodutível

for i in range(n_permutacoes):
    y_embaralhado = rng.permutation(y)

    y_real_perm, y_previsto_perm, _ = rodar_logo(
        lambda: PLSRegression(n_components=melhor_n), X, y_embaralhado, grupos
    )

    r2_perm = r2_score(y_real_perm, y_previsto_perm)
    r2_permutados.append(r2_perm)

r2_permutados = np.array(r2_permutados)

# p-valor: em quantas permutações o R2 do acaso foi igual ou maior que o real?
p_valor = np.mean(r2_permutados >= r2_reais)

print(f"R² real do modelo: {r2_reais:.3f}")
print(f"R² médio das permutações (acaso): {r2_permutados.mean():.3f}")
print(f"p-valor: {p_valor:.4f}")

if p_valor < 0.05:
    print("=> O modelo é significativamente melhor do que o esperado por acaso (p < 0.05).")
else:
    print("=> Não há evidência suficiente de que o modelo supere o acaso (p >= 0.05).")

# Gráfico: onde o R2 real cai em relação à distribuição do acaso
fig_perm = px.histogram(
    x=r2_permutados, nbins=40,
    labels={"x": "R² obtido com Y embaralhado (acaso)"},
    title=f"Teste de permutação ({n_permutacoes} repetições) - p-valor = {p_valor:.4f} (p < 0,0001)"
)
fig_perm.add_vline(
    x=r2_reais, line_dash="dash", line_color="red",
    annotation_text=f"R² real = {r2_reais:.3f}", annotation_position="top"
)
fig_perm.show()
exportar_para_artigo(fig_perm, nome_arquivo="fig04_teste_permutacao.png")

## 5. Validação aninhada (nested LOGO)

Como o número de componentes do PLSR foi escolhido observando o próprio resultado da validação Leave-One-Group-Out, existe um viés de seleção: o R² relatado anteriormente pode estar levemente otimista, já que a mesma validação usada para escolher o hiperparâmetro também foi usada para avaliá-lo. Para corrigir isso, foi implementada uma validação aninhada, em que a escolha do número de componentes é feita usando apenas os dados de treino de cada rodada externa, e o desempenho final é medido em uma réplica que nunca participou dessa escolha.


In [ ]:

def rodar_logo_aninhado(X_raw, y_alvo, grupos_cv, componentes_candidatos=[1,2,3,4]):
    logo_externo = LeaveOneGroupOut()
    y_real, y_previsto, n_escolhidos = [], [], []

    for idx_treino, idx_teste in logo_externo.split(X_raw, y_alvo, groups=grupos_cv):
        X_treino_ext, y_treino_ext = X_raw[idx_treino], y_alvo[idx_treino]
        grupos_treino_ext = grupos_cv[idx_treino]

        # Escolhe n_componentes usando SÓ os dados de treino (LOGO interno)
        melhor_r2_interno, melhor_n_interno = -np.inf, componentes_candidatos[0]
        for n in componentes_candidatos:
            y_r_int, y_p_int, _ = rodar_logo(
                lambda: PLSRegression(n_components=n), X_treino_ext, y_treino_ext, grupos_treino_ext
            )
            r2_int = r2_score(y_r_int, y_p_int)
            if r2_int > melhor_r2_interno:
                melhor_r2_interno, melhor_n_interno = r2_int, n

        # Treina com o n escolhido e testa na réplica externa (nunca vista)
        scaler_fold = StandardScaler()
        X_tr = scaler_fold.fit_transform(X_treino_ext)
        X_te = scaler_fold.transform(X_raw[idx_teste])
        modelo = PLSRegression(n_components=melhor_n_interno)
        modelo.fit(X_tr, y_treino_ext)
        pred = np.ravel(modelo.predict(X_te))

        y_real.extend(y_alvo[idx_teste].tolist())
        y_previsto.extend(pred.tolist())
        n_escolhidos.append(melhor_n_interno)

    return np.array(y_real), np.array(y_previsto), n_escolhidos

y_real_nested, y_previsto_nested, n_escolhidos = rodar_logo_aninhado(X, y, grupos)
r2_nested = r2_score(y_real_nested, y_previsto_nested)
rmse_nested = np.sqrt(mean_squared_error(y_real_nested, y_previsto_nested))
print(f"R² (validação aninhada, sem viés de seleção): {r2_nested:.3f}")
print(f"RMSE (validação aninhada): {rmse_nested:.3f}")
print(f"n_componentes escolhidos por fold: {n_escolhidos}")

idx_teste_nested = np.concatenate([ix for _, ix in LeaveOneGroupOut().split(X, y, groups=grupos)])

## 6. Comparação com outros modelos

Para contextualizar o desempenho do PLSR, o mesmo esquema de validação Leave-One-Group-Out foi aplicado a outros modelos de regressão.

A título de comparação, o resultado abaixo mostra primeiro os modelos sem nenhum cuidado adicional de ajuste, sendo o Ridge com alpha fixo (1.0) e Random Forest sem restrição de complexidade. Esses resultados tendem a ser artificialmente otimistas, já que nenhum hiperparâmetro foi escolhido por validação e o Random Forest tem liberdade para se ajustar demais aos dados de treino.

Nas subseções seguintes (6.1 e 6.2), esses dois modelos são refeitos com os devidos cuidados: o alpha do Ridge passa a ser escolhido por validação, do mesmo modo que o número de componentes do PLSR foi ajustado na seção 5, e o Random Forest tem sua complexidade limitada de propósito. A seção 6.3 então repete essa comparação com as versões corrigidas, permitindo visualizar diretamente o efeito da validação adequada sobre o desempenho aparente de cada modelo.

In [ ]:

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

comparacao_resultados = []

modelos_builders = {
    "PLSR": lambda: PLSRegression(n_components=melhor_n),
    "Ridge": lambda: Ridge(alpha=1.0),
    "Random Forest": lambda: RandomForestRegressor(n_estimators=200, random_state=42),
}

for nome_modelo, builder in modelos_builders.items():
    y_real_modelo, y_previsto_modelo, _ = rodar_logo(builder, X, y, grupos)

    r2_modelo = r2_score(y_real_modelo, y_previsto_modelo)
    rmse_modelo = np.sqrt(mean_squared_error(y_real_modelo, y_previsto_modelo))
    comparacao_resultados.append({
        "modelo": nome_modelo, "R2_LOGO": r2_modelo, "RMSE_LOGO": rmse_modelo
    })

comparacao_df = pd.DataFrame(comparacao_resultados).sort_values("R2_LOGO", ascending=False)
print("\nComparação de modelos (mesma validação Leave-One-Group-Out):")
print(comparacao_df)

fig_comparacao = px.bar(
    comparacao_df, x="modelo", y="R2_LOGO",
    title="Comparação de modelos - R² na validação (Leave-One-Group-Out)",
    text_auto=".3f"
)
fig_comparacao.show()
exportar_para_artigo(fig_comparacao, nome_arquivo="fig05_comparacao_modelos.png")

### 6.1 Ridge com alpha ajustado por validação

A comparação inicial usava Ridge com alpha fixo (1.0), o que não é uma comparação justa com o PLSR, que teve seu número de componentes escolhido por validação (seção 5). Aqui o alpha é escolhido dentro de um LOGO interno (usando só os dados de treino de cada rodada externa), seguindo a mesma lógica da validação aninhada aplicada ao PLSR.

In [ ]:
from sklearn.linear_model import Ridge

def rodar_logo_aninhado_ridge(X_raw, y_alvo, grupos_cv, alphas_candidatos=[0.1, 1, 10, 50, 100]):
    logo_externo = LeaveOneGroupOut()
    y_real, y_previsto, alphas_escolhidos = [], [], []

    for idx_treino, idx_teste in logo_externo.split(X_raw, y_alvo, groups=grupos_cv):
        X_treino_ext, y_treino_ext = X_raw[idx_treino], y_alvo[idx_treino]
        grupos_treino_ext = grupos_cv[idx_treino]

        # Escolhe alpha usando SÓ os dados de treino (LOGO interno)
        melhor_r2_interno, melhor_alpha_interno = -np.inf, alphas_candidatos[0]
        for a in alphas_candidatos:
            y_r_int, y_p_int, _ = rodar_logo(
                lambda a=a: Ridge(alpha=a), X_treino_ext, y_treino_ext, grupos_treino_ext
            )
            r2_int = r2_score(y_r_int, y_p_int)
            if r2_int > melhor_r2_interno:
                melhor_r2_interno, melhor_alpha_interno = r2_int, a

        # Treina com o alpha escolhido e testa na réplica externa (nunca vista)
        scaler_fold = StandardScaler()
        X_tr = scaler_fold.fit_transform(X_treino_ext)
        X_te = scaler_fold.transform(X_raw[idx_teste])
        modelo = Ridge(alpha=melhor_alpha_interno)
        modelo.fit(X_tr, y_treino_ext)
        pred = np.ravel(modelo.predict(X_te))

        y_real.extend(y_alvo[idx_teste].tolist())
        y_previsto.extend(pred.tolist())
        alphas_escolhidos.append(melhor_alpha_interno)

    return np.array(y_real), np.array(y_previsto), alphas_escolhidos

y_real_ridge, y_previsto_ridge, alphas_escolhidos = rodar_logo_aninhado_ridge(X, y, grupos)
r2_ridge_nested = r2_score(y_real_ridge, y_previsto_ridge)
rmse_ridge_nested = np.sqrt(mean_squared_error(y_real_ridge, y_previsto_ridge))
print(f"R² Ridge (alpha ajustado, validação aninhada): {r2_ridge_nested:.3f}")
print(f"RMSE Ridge (alpha ajustado, validação aninhada): {rmse_ridge_nested:.3f}")
print(f"alphas escolhidos por fold: {alphas_escolhidos}")

### 6.2 Random Forest como modelo exploratório não linear

O Random Forest é incluído para verificar se relações não lineares entre os atributos trariam ganho sobre os modelos lineares. Como o conjunto tem apenas 30 observações vindas de 5 réplicas, a complexidade do modelo foi limitada de propósito (árvores rasas, poucas variáveis candidatas por divisão, folhas mínimas maiores), para reduzir o risco de o modelo memorizar particularidades das amostras de treino em vez de aprender um padrão generalizável. Esse modelo deve ser lido como exploratório, não como concorrente direto do PLSR/Ridge.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def construir_rf():
    return RandomForestRegressor(
        n_estimators=200,
        max_depth=3,            # árvores rasas
        max_features="sqrt",    # poucas variáveis por divisão
        min_samples_leaf=3,     # regularização extra
        random_state=42,
    )

y_real_rf, y_previsto_rf, _ = rodar_logo(construir_rf, X, y, grupos)
r2_rf = r2_score(y_real_rf, y_previsto_rf)
rmse_rf = np.sqrt(mean_squared_error(y_real_rf, y_previsto_rf))
print(f"R² Random Forest (LOGO, complexidade limitada): {r2_rf:.3f}")
print(f"RMSE Random Forest (LOGO, complexidade limitada): {rmse_rf:.3f}")

### 6.3 Regressão linear sem regularização (OLS)

Como complemento à comparação, foi incluída a regressão linear múltipla
sem nenhum controle de complexidade (Ordinary Least Squares, OLS),
utilizando o conjunto completo de 23 atributos. Diferente da abordagem
univariada empregada na etapa inicial do projeto, que ajustava uma
regressão linear simples para cada atributo isoladamente em função do
tempo, este modelo utiliza todos os atributos simultaneamente, mas sem
qualquer mecanismo de penalização sobre os coeficientes.

Sob o mesmo esquema de validação Leave-One-Group-Out utilizado nos
demais modelos, o resultado evidencia de forma direta os riscos de
ajustar um modelo sem controle de complexidade em um cenário de alta
dimensionalidade relativa ao número de amostras: com 23 atributos e
aproximadamente 24 observações de treino por rodada de validação, o
modelo dispõe de liberdade suficiente para se ajustar quase perfeitamente
aos dados de treino, à custa de generalização praticamente nula para
dados não vistos.

In [ ]:
from sklearn.linear_model import LinearRegression

y_real_ols, y_previsto_ols, _ = rodar_logo(LinearRegression, X, y, grupos)
r2_ols = r2_score(y_real_ols, y_previsto_ols)
rmse_ols = np.sqrt(mean_squared_error(y_real_ols, y_previsto_ols))
print(f"R² Regressão Linear (OLS, sem regularização): {r2_ols:.3f}")
print(f"RMSE Regressão Linear (OLS, sem regularização): {rmse_ols:.3f}")

In [ ]:
idx_teste_ols = np.concatenate([ix for _, ix in LeaveOneGroupOut().split(X, y, groups=grupos)])
for rep in np.unique(grupos):
    mask = grupos[idx_teste_ols] == rep
    print(rep, r2_score(y_real_ols[mask], y_previsto_ols[mask]))

In [ ]:
modelo_teste = LinearRegression()
modelo_teste.fit(X[grupos != 1], y[grupos != 1])  # treina deixando a réplica 1 de fora, por exemplo
print(np.abs(modelo_teste.coef_).sort_values(ascending=False) if hasattr(modelo_teste.coef_, 'sort_values') else np.sort(np.abs(modelo_teste.coef_))[::-1])

A instabilidade do modelo não se distribuiu uniformemente entre as
réplicas: enquanto os folds referentes às réplicas 1 e 2 apresentaram
desempenho relativamente aceitável (R² = 0,917 e 0,945, respectivamente),
o fold referente à réplica 3 apresentou desempenho extremamente negativo
(R² = -161,31), responsável isoladamente pela maior parte do resultado
médio observado. Essa concentração de erro é consistente com o
comportamento esperado de modelos de regressão linear sem regularização
em cenários de quase-colinearidade: a inspeção dos coeficientes do modelo
revelou valores de magnitude muito superior ao esperado para atributos
padronizados (coeficiente máximo de 31,90, frente aos valores tipicamente
observados em modelos regularizados), evidenciando que o modelo compensa
a correlação entre atributos com coeficientes de sinais opostos e grande
magnitude, tornando as previsões extremamente sensíveis a pequenas
variações nos dados de teste.

### 6.4 Comparação final entre os modelos

Com o PLSR (componentes escolhidos por validação, seção 5), o Ridge (alpha escolhido por validação, seção 6.1) e o Random Forest (complexidade limitada, seção 6.2), a comparação abaixo passa a ser justa entre os três: nenhum modelo está usando um hiperparâmetro fixo arbitrário ou uma flexibilidade excessiva não controlada.

In [ ]:
comparacao_final = pd.DataFrame([
    {"modelo": "PLSR", "R2_LOGO": r2_nested, "RMSE_LOGO": rmse_nested},
    {"modelo": "Ridge (alpha ajustado)", "R2_LOGO": r2_ridge_nested, "RMSE_LOGO": rmse_ridge_nested},
    {"modelo": "Random Forest", "R2_LOGO": r2_rf, "RMSE_LOGO": rmse_rf},
]).sort_values("R2_LOGO", ascending=False)

print("\nComparação final de modelos (mesma validação Leave-One-Group-Out):")
print(comparacao_final)

fig_comparacao_final = px.bar(
    comparacao_final, x="modelo", y="R2_LOGO",
    title="Comparação final de modelos (hiperparâmetros ajustados/limitados)",
    text_auto=".3f"
)
fig_comparacao_final.show()
exportar_para_artigo(fig_comparacao_final, nome_arquivo="fig06_comparacao_final.png")

### 6.5 Efeito visual da ausência de regularização

In [ ]:
comparacao_rmse = pd.DataFrame([
    {"modelo": "Baseline (média)", "RMSE": 4.072},
    {"modelo": "Random Forest", "RMSE": rmse_rf},
    {"modelo": "Ridge (alpha ajustado)", "RMSE": rmse_ridge_nested},
    {"modelo": "PLSR", "RMSE": rmse_nested},
    {"modelo": "Regressão Linear (OLS)", "RMSE": rmse_ols},
])

fig_rmse_log = px.bar(
    comparacao_rmse, x="modelo", y="RMSE",
    title="RMSE por modelo (escala logarítmica, ilustrando o efeito da ausência de regularização)",
    text_auto=".2f",
    log_y=True
)
fig_rmse_log.show()
exportar_para_artigo(fig_rmse_log, nome_arquivo="fig07_rmse_log.png")

O gráfico em escala logarítmica evidencia visualmente a magnitude do
efeito da ausência de regularização: enquanto o modelo de referência
(baseline da média), o Random Forest, o Ridge ajustado e o PLSR
concentram-se numa faixa relativamente próxima de RMSE (entre
aproximadamente 0,90 e 4,07), a regressão linear sem regularização
apresenta erro de 23,32, cerca de 6 vezes maior que o próprio baseline
que não utiliza nenhum atributo. Esse resultado situa a regressão linear
sem controle de complexidade não apenas como o pior modelo avaliado, mas
como um modelo pior do que a completa ausência de informação preditiva,
o que reforça, de forma direta e quantitativa, a importância de mecanismos
de regularização e de redução de dimensionalidade em cenários com muitos
atributos e poucas amostras.

É importante ressaltar que a regressão linear sem regularização não foi
incluída nesta comparação como candidata a modelo final, mas como
referência de controle, isolando o efeito da regularização presente nos
demais modelos avaliados. Diferentemente do Ridge, do PLSR e do Random
Forest, cujos hiperparâmetros foram ajustados por validação ou limitados
deliberadamente, a regressão linear múltipla não dispõe de nenhum
mecanismo equivalente de controle, sendo, por definição, o caso extremo
dessa comparação. Nesse sentido, o desempenho drasticamente inferior
observado não representa uma limitação do método em si, mas confirma
empiricamente a necessidade dos cuidados metodológicos adotados nos
demais modelos deste trabalho.

## Verificações adicionais

Por fim, alguns testes complementares foram feitos para reforçar a leitura dos resultados anteriores.

**RMSE de referência (baseline).** Como termo de comparação, calculamos o RMSE de um modelo ingênuo que prevê sempre a média do tempo para todas as amostras, independentemente dos atributos medidos.

In [ ]:
rmse_baseline = np.sqrt(mean_squared_error(y, [y.mean()]*len(y)))
print(f"RMSE só chutando a média: {rmse_baseline:.3f}")

**R² por réplica.** Também é possível olhar o erro do modelo final separado por réplica, em vez de por período de tempo, para verificar se alguma amostra específica é sistematicamente mais difícil de prever do que as demais.

In [ ]:
for rep in np.unique(grupos):
    mask = grupos[idx_teste_nested] == rep
    print(rep, r2_score(y_real_nested[mask], y_previsto_nested[mask]))

**Correlação de cada atributo com o tempo.** Uma checagem simples e direta é observar quais atributos, isoladamente, mais se correlacionam com o tempo de maturação, o que ajuda a confirmar (ou não) a relevância dos atributos apontados pelo VIP.

In [ ]:
print(df[colunas_atributos].corrwith(df["Tempo_num"]).sort_values(key=abs, ascending=False).head(5))

**Modelo com um único atributo.** Como teste final, ajustamos um PLSR utilizando apenas o atributo "Straw odor", um dos mais correlacionados com o tempo, para verificar se um modelo bem mais simples consegue capturar sozinho parte do sinal temporal.

In [ ]:
X_simples = df[["Straw odor"]].values
y_real_s, y_previsto_s, _ = rodar_logo(lambda: PLSRegression(n_components=1), X_simples, y, grupos)
print("R² só com Straw odor:", r2_score(y_real_s, y_previsto_s))